In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, re, json, shutil, hashlib
from collections import Counter
import pandas as pd
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

RAW_DIR = "/content/raw_dataset"
SEED = 42

In [3]:
shutil.rmtree(RAW_DIR, ignore_errors=True)
shutil.copytree("/content/drive/MyDrive/IE7615 Deep Learning for AI", RAW_DIR)
all_files = sorted(os.path.join(r, f) for r, _, fs in os.walk(RAW_DIR) for f in fs)
print("Files:", len(all_files))

shutil.make_archive("/content/raw_dataset", "zip", RAW_DIR)
os.makedirs("/content/drive/MyDrive/IE7615", exist_ok=True)
shutil.copy("/content/raw_dataset.zip", "/content/drive/MyDrive/IE7615/raw_dataset.zip")

Files: 7345


'/content/drive/MyDrive/IE7615/raw_dataset.zip'

In [4]:
OBJ_RE = re.compile(r"(OBJ\d{3,})", re.IGNORECASE)
IMAGE_EXT = {".jpg", ".jpeg", ".png"}
IGNORE = {".ds_store", "desktop.ini", "thumbs.db"}

def label_from_path(path):
    for part in reversed(os.path.relpath(path, RAW_DIR).split(os.sep)[:-1]):
        m = OBJ_RE.search(part)
        if m:
            return m.group(1).upper()
    return "unknown"

valid, corrupted, non_image = [], [], []
for path in tqdm(all_files, desc="Step 2: validating"):
    name = os.path.basename(path)
    if name.lower() in IGNORE or name.startswith("._"):
        continue
    if os.path.splitext(name)[1].lower() not in IMAGE_EXT:
        non_image.append(path)
        continue
    try:
        with Image.open(path) as im:
            w, h, mode = im.size[0], im.size[1], im.mode
            if im.format == "JPEG":
                im.draft("RGB", (448, 448))
            im.load()
        valid.append({"path": path, "label": label_from_path(path),
                      "orig_w": w, "orig_h": h, "orig_mode": mode})
    except Exception as e:
        corrupted.append((path, f"{type(e).__name__}: {e}"))

df = pd.DataFrame(valid)
print(f"Valid images:     {len(df)}")
print(f"Corrupted:        {len(corrupted)}")
print(f"Non-image files:  {len(non_image)}")
for p, err in corrupted:
    print("  corrupted:", os.path.relpath(p, RAW_DIR), "->", err)
assert (df.label == "unknown").sum() == 0, df[df.label == "unknown"].path.tolist()

Step 2: validating:   0%|          | 0/7345 [00:00<?, ?it/s]

Valid images:     7343
Corrupted:        0
Non-image files:  1


In [5]:
BG_RE = re.compile(r"(?:^|[_\-\s])(?:bg|background|backgrnd|back_ground|noobj|no_obj)(?:[_\-\s.]|$)", re.IGNORECASE)
MANUAL_BG = set()

df["fname"] = df.path.map(os.path.basename)
df["is_bg"] = df.fname.str.contains(BG_RE) | df.fname.isin(MANUAL_BG)

print(f"Images that look like background shots: {df.is_bg.sum()}")
print("\nExample background filenames:")
print(df[df.is_bg].fname.head(10).to_string(index=False))

bg_per_class = df.groupby("label").is_bg.sum()
no_bg = bg_per_class[bg_per_class == 0].index.tolist()
print(f"\nClasses where NO background file was detected: {len(no_bg)}")
for lab in no_bg[:5]:
    names = df[df.label == lab].fname.sort_values().tolist()
    print(f"  {lab}: {names[:3]} ... {names[-3:]}")

Images that look like background shots: 379

Example background filenames:
OBJ001_bg_001.jpg
OBJ001_bg_002.jpg
OBJ001_bg_003.jpg
OBJ001_bg_004.jpg
OBJ001_bg_005.jpg
OBJ002_bg_001.jpg
OBJ002_bg_002.jpg
OBJ002_bg_003.jpg
OBJ002_bg_004.jpg
OBJ002_bg_005.jpg

Classes where NO background file was detected: 2
  OBJ004: ['OBJ004_001.jpg', 'OBJ004_002.jpg', 'OBJ004_003.jpg'] ... ['OBJ004_098.jpg', 'OBJ004_099.jpg', 'OBJ004_100.jpg']
  OBJ013: ['OBJ013_001.jpg', 'OBJ013_002.jpg', 'OBJ013_003.jpg'] ... ['OBJ013_BG004.jpg', 'OBJ013_BG005.jpg', 'OBJ013_BG006.jpg']


In [6]:
bg_removed = df[df.is_bg]
df = df[~df.is_bg].drop(columns=["fname", "is_bg"]).reset_index(drop=True)

print(f"Background images removed: {len(bg_removed)}")
print(f"Images remaining: {len(df)}")
print(f"Classes: {df.label.nunique()}")

Background images removed: 379
Images remaining: 6964
Classes: 73


In [7]:
assert not df.path.map(os.path.basename).str.contains(BG_RE).any(), "background images still in df, run cells in order"
def md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

df["md5"] = [md5(p) for p in tqdm(df.path, desc="Step 3: hashing")]

dup_mask = df.duplicated("md5", keep="first")
dups = df[dup_mask]
first_label = df[~dup_mask].set_index("md5").label
cross_class = dups[dups.label != dups.md5.map(first_label)]

print(f"Exact duplicates removed: {len(dups)}")
print(f"  of which the same image appears in two different classes: {len(cross_class)}")
if len(dups):
    print("\nDuplicates per class:")
    print(dups.label.value_counts().to_string())

df = df[~dup_mask].reset_index(drop=True)
print(f"\nImages remaining: {len(df)}")

Step 3: hashing:   0%|          | 0/6964 [00:00<?, ?it/s]

Exact duplicates removed: 16
  of which the same image appears in two different classes: 0

Duplicates per class:
label
OBJ023    6
OBJ062    4
OBJ011    2
OBJ022    1
OBJ028    1
OBJ029    1
OBJ066    1

Images remaining: 6948


In [8]:
IMG_SIZE = 224
RESIZE_MODE = "pad"   # "pad" = keep shape, add black borders | "stretch" = force to 224x224
STAGE4 = "/content/stage4_resized"
shutil.rmtree(STAGE4, ignore_errors=True)

def resize(im):
    if RESIZE_MODE == "stretch":
        return im.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
    im = ImageOps.contain(im, (IMG_SIZE, IMG_SIZE), Image.LANCZOS)
    fill = (0,) * len(im.getbands())
    canvas = Image.new(im.mode, (IMG_SIZE, IMG_SIZE), fill)
    canvas.paste(im, ((IMG_SIZE - im.width) // 2, (IMG_SIZE - im.height) // 2))
    return canvas

stage4_paths, name_count = [], Counter()
for path, label in tqdm(zip(df.path, df.label), total=len(df), desc="Step 4: resizing"):
    with Image.open(path) as im:
        if im.format == "JPEG":
            im.draft(im.mode, (IMG_SIZE * 2, IMG_SIZE * 2))
        im.load()
        im = ImageOps.exif_transpose(im)
        if im.mode == "P":
            im = im.convert("RGBA")
        im = resize(im)
    stem = re.sub(r"[^\w\-]+", "_", os.path.splitext(os.path.basename(path))[0])
    name_count[(label, stem)] += 1
    if name_count[(label, stem)] > 1:
        stem = f"{stem}_{name_count[(label, stem)]}"
    out = os.path.join(STAGE4, label, stem + ".png")
    os.makedirs(os.path.dirname(out), exist_ok=True)
    im.save(out)
    stage4_paths.append(out)
df["stage4_path"] = stage4_paths

was_resized = (df.orig_w != IMG_SIZE) | (df.orig_h != IMG_SIZE)
print(f"Images resized (were not 224x224): {was_resized.sum()}")
for lab, g in df[was_resized].groupby("label"):
    sizes = Counter(f"{w}x{h}" for w, h in zip(g.orig_w, g.orig_h)).most_common(3)
    print(f"  {lab}: {len(g)} images, original sizes {sizes}")

Step 4: resizing:   0%|          | 0/6948 [00:00<?, ?it/s]

Images resized (were not 224x224): 2
  OBJ003: 1 images, original sizes [('224x192', 1)]
  OBJ023: 1 images, original sizes [('225x224', 1)]


In [9]:
STAGE5 = "/content/stage5_rgb"
shutil.rmtree(STAGE5, ignore_errors=True)

def to_rgb(im):
    if im.mode in ("RGBA", "LA"):
        im = im.convert("RGBA")
        bg = Image.new("RGB", im.size, (255, 255, 255))
        bg.paste(im, mask=im.split()[-1])
        return bg
    return im.convert("RGB")

stage5_paths = []
for p, label in tqdm(zip(df.stage4_path, df.label), total=len(df), desc="Step 5: RGB"):
    with Image.open(p) as im:
        im = to_rgb(im)
    out = os.path.join(STAGE5, label, os.path.splitext(os.path.basename(p))[0] + ".jpg")
    os.makedirs(os.path.dirname(out), exist_ok=True)
    im.save(out, "JPEG", quality=95)
    stage5_paths.append(out)
df["stage5_path"] = stage5_paths
shutil.rmtree(STAGE4)

print("Original colour modes:", dict(Counter(df.orig_mode)))
print(f"Converted to RGB: {(df.orig_mode != 'RGB').sum()} images")

Step 5: RGB:   0%|          | 0/6948 [00:00<?, ?it/s]

Original colour modes: {'RGB': 6948}
Converted to RGB: 0 images


In [10]:
OUT_DIR = "/content/processed_224"
shutil.rmtree(OUT_DIR, ignore_errors=True)

counts = df.label.value_counts()
tiny = counts[counts < 10].index
if len(tiny):
    print("Classes with <10 images go fully to train:", list(tiny))
big = df[~df.label.isin(tiny)]

train_df, temp_df = train_test_split(big, test_size=0.30, stratify=big.label, random_state=SEED)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, stratify=temp_df.label, random_state=SEED)
train_df = pd.concat([train_df, df[df.label.isin(tiny)]])

df["split"] = ""
df.loc[train_df.index, "split"] = "train"
df.loc[val_df.index, "split"] = "val"
df.loc[test_df.index, "split"] = "test"

final_paths = []
for _, r in tqdm(df.iterrows(), total=len(df), desc="Step 7: splitting"):
    dst = os.path.join(OUT_DIR, r.split, r.label, os.path.basename(r.stage5_path))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.move(r.stage5_path, dst)
    final_paths.append(dst)
df["final_path"] = final_paths
shutil.rmtree(STAGE5)

print(df.split.value_counts().to_string())
print(f"Classes in train/val/test: {train_df.label.nunique()} / {val_df.label.nunique()} / {test_df.label.nunique()}")

bad = 0
for p in df.final_path:
    with Image.open(p) as im:
        bad += (im.size != (IMG_SIZE, IMG_SIZE)) or (im.mode != "RGB")
print("Images not 224x224 RGB:", bad)
print("Same image in more than one split:", df.groupby("md5").split.nunique().gt(1).sum())

df[["path", "label", "orig_w", "orig_h", "orig_mode", "md5", "split", "final_path"]] \
    .to_csv(f"{OUT_DIR}/manifest.csv", index=False)
report = {
    "total_files": len(all_files), "corrupted": len(corrupted), "non_image": len(non_image),"background_removed": len(bg_removed),
    "duplicates_removed": len(dups), "cross_class_duplicates": len(cross_class),
    "resized_from_non_224": int(was_resized.sum()),
    "converted_to_rgb": int((df.orig_mode != "RGB").sum()),
    "final_images": len(df), "classes": int(df.label.nunique()),
    "split": df.split.value_counts().to_dict(), "seed": SEED, "resize_mode": RESIZE_MODE,
}
json.dump(report, open(f"{OUT_DIR}/preprocessing_report.json", "w"), indent=2)
print(json.dumps(report, indent=2))
classes = sorted(df.label.unique())
json.dump({c: i for i, c in enumerate(classes)}, open(f"{OUT_DIR}/class_to_idx.json", "w"), indent=2)
shutil.make_archive("/content/processed_224", "zip", OUT_DIR)
shutil.copy("/content/processed_224.zip", "/content/drive/MyDrive/IE7615/processed_224.zip")
print("Saved processed dataset to Drive")

Step 7: splitting:   0%|          | 0/6948 [00:00<?, ?it/s]

split
train    4863
test     1043
val      1042
Classes in train/val/test: 73 / 73 / 73
Images not 224x224 RGB: 0
Same image in more than one split: 0
{
  "total_files": 7345,
  "corrupted": 0,
  "non_image": 1,
  "background_removed": 379,
  "duplicates_removed": 16,
  "cross_class_duplicates": 0,
  "resized_from_non_224": 2,
  "converted_to_rgb": 0,
  "final_images": 6948,
  "classes": 73,
  "split": {
    "train": 4863,
    "test": 1043,
    "val": 1042
  },
  "seed": 42,
  "resize_mode": "pad"
}
Saved processed dataset to Drive


In [11]:
print(bg_removed.groupby("label").size().loc[lambda s: s != 5])

label
OBJ005     9
OBJ009    11
OBJ014     7
OBJ017     7
OBJ025     6
OBJ049     6
OBJ053    10
OBJ063     6
OBJ069     7
dtype: int64
